In [1]:
import os

In [2]:
%pwd

'/workspaces/MLOps-Laptop-Prediction-System/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/workspaces/MLOps-Laptop-Prediction-System'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: str
    preprocessor: str
    model_path: str
    metric_file_name: str
    target_column: str

In [6]:
from Laptop_Price_Prediction_System.constants import *
from Laptop_Price_Prediction_System.utils.common import read_yaml, create_directories, save_json

In [7]:
class ConfigurationManager:
    def __init__(self, config_filepath=Config_Filepath, params_filepath=Params_Filepath, schema_filepath=Schema_Filepath):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            preprocessor=config.preprocessor_obj_file_path,
            model_path=config.model_path,
            metric_file_name=config.metric_file_name,
            target_column=schema.name
        )

        return model_evaluation_config

In [ ]:
import numpy as np
import pandas as pd
from Laptop_Price_Prediction_System import logger
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

In [9]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config
    
    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)

        return rmse, mae, r2
    
    def evaluation(self):
        test_data = pd.read_csv(self.config.test_data_path)

        preprocessor = joblib.load(self.config.preprocessor)
        model = joblib.load(self.config.model_path)


        test_X = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[self.config.target_column]

        X_preprocessor = preprocessor.transform(test_X)

        predicted_qualities = model.predict(X_preprocessor)

        logger.info("Starting Model Evaluation")
        rmse, mae, r2_score = self.eval_metrics(test_y, predicted_qualities)

        scores = {"rmse": rmse, "mae": mae, "r2_score": r2_score}
        save_json(path=Path(self.config.metric_file_name), data=scores)

In [ ]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.evaluation()
except Exception as e:
    raise e

[2026-07-10 11:49:03,859 : INFO : common : Loaded file: config/config.yaml]
[2026-07-10 11:49:03,862 : INFO : common : Loaded file: params.yaml]
[2026-07-10 11:49:03,866 : INFO : common : Loaded file: schema.yaml]
[2026-07-10 11:49:03,868 : INFO : common : Created directory: artifacts]
[2026-07-10 11:49:03,869 : INFO : common : Created directory: artifacts/model_evaluation]
[2026-07-10 11:49:04,480 : INFO : 295743173 : Starting Model Evaluation]
[2026-07-10 11:49:04,487 : INFO : common : JSON file saved at artifacts/model_evaluation/metrics.json]
